# Titanic Survival Prediction using Iguanas with Feature Engineering

This notebook demonstrates a complete end-to-end example of using **Iguanas** for rule-based classification on the Kaggle Titanic dataset, with advanced feature engineering using the **Gators** library.

The workflow includes:
1. Loading and exploring the data
2. **Feature engineering using Gators** (new columns, transformations, encoding)
3. Generating candidate rules using XGBoost
4. Filtering and selecting high-quality rules
5. Combining rules using different strategies
6. Generating predictions for submission

## 1. Import Libraries

In [9]:
import polars as  pl
import numpy as np
from xgboost import XGBClassifier

from gators.pipeline import Pipeline
from gators.encoders import RareCategoryEncoder, WOEEncoder
from gators.discretizers import CustomDiscretizer
from gators.data_cleaning import DropColumns, CastColumns, RenameColumns
from gators.feature_generation import (
    IsNull, 
    MathFeatures, 
    ConditionFeatures, 
    ScalarMathFeatures
)
from gators.imputers import StringImputer, NumericImputer
from gators.feature_generation_str import (
    Length,
    SplitExtract,
    ExtractSubstring,
)

from iguanas.rule_generation import rule_grid_search_parallel_scales
from iguanas.rule_evaluation import apply_rules
from iguanas.metrics import compute_metrics
from iguanas.rule_combination import (
    combine_rules_cumulative,
    combine_rules_greedy,
    combine_rules_beam_search,
    combine_rules_a_star
)
from iguanas.rule_selection import filter_correlated_rules
from iguanas.rule_analysis import generate_rule_performance_report

## 2. Load and Prepare Data

Load the Titanic training data and separate features from the target variable (Survived).

In [2]:
train = pl.read_csv('../../../kaggle/titanic/train.csv').drop('PassengerId')
X_train = train.drop('Survived')
y_train = train['Survived']

## 3. Feature Engineering with Gators

Build a comprehensive feature engineering pipeline using the **Gators** library. This pipeline will:
- Create missing value indicators for Age and Cabin
- Extract string features (name titles, cabin deck, ticket length)
- Calculate family size and fare per person
- Create categorical bins for age
- Generate feature interactions
- Apply Weight of Evidence (WOE) encoding for all categorical variables

**Key transformations in this pipeline:**

1. **Missing value handling**: Create indicators for missing Age/Cabin, then impute
2. **Feature extraction**: Extract passenger titles from names, cabin deck letters, ticket lengths
3. **Feature creation**: Calculate family size, fare per person, traveling alone indicator
4. **Discretization**: Convert continuous Age into categorical bins
5. **Interactions**: Create combinations of Pclass, Age, CabinDeck, and Embarked
6. **Encoding**: Apply WOE encoding to convert all categorical features to numeric values

In [3]:
# Define the feature engineering pipeline
steps = [
    # Create missing value indicators
    ('IsNull', IsNull(subset=['Age', 'Cabin'])),
    
    # String feature engineering
    ('Length', Length(subset=['Ticket'])),
    ('SplitExtractName', SplitExtract(subset=['Name'], by=', ', n=1)),
    ('SplitExtractTitle', SplitExtract(subset=['Name__split_,__1'], by='.', n=0)),
    
    # Calculate family size (SibSp + Parch + 1)
    ('MathFeatures', MathFeatures(
        groups=[['SibSp', 'Parch']],
        operations=['sum'],
        new_column_names=['Dummy']
    )),
    ('ScalarMathFeatures', ScalarMathFeatures(
        operations=[{'column': 'Dummy_sum', 'op': '+', 'scalar': 1}],
        new_column_names=["FamilySize"]
    )),
    
    # Extract cabin deck (first letter of cabin)
    ('ExtractSubstring', ExtractSubstring(subset=['Cabin'], start=0, end=1)),
    
    # Rename for clarity
    ('RenameColumns', RenameColumns(column_mapping={
        'Name__split_,__1__split_._0': 'Title',
        'Cabin__start0_end1': 'CabinDeck'
    })),
    
    # Handle rare categories (group infrequent values)
    ('RareCategoryEncoder', RareCategoryEncoder(min_count=0.01)),
    
    # Calculate fare per person
    ('MathFeatures2', MathFeatures(
        groups=[['Fare', 'FamilySize']],
        operations=['div'],
        new_column_names=['FarePerPerson']
    )),
    
    # Create 'traveling alone' indicator
    ('ConditionFeatures', ConditionFeatures(
        conditions=[{"column": "FamilySize", "op": ">", "value": 1}],
        new_column_names=['IsAlone']
    )),
    
    # Drop raw columns no longer needed
    ('DropColumns', DropColumns(subset=['Cabin', 'Ticket', 'Dummy_sum'])),
    
    # Impute missing values
    ('NumericImputer', NumericImputer(strategy='mean')),
    ('StringImputer', StringImputer(strategy='constant', value='MISSING')),
    
    # Discretize age into bins
    ('CustomDiscretizer', CustomDiscretizer(
        bins={'Age': [0, 12, 18, 35, 60, 100]},
        inplace=True
    )),
    
    # Convert passenger class to categorical
    ('CastColumns', CastColumns(subset=["Pclass"], dtype=pl.String)),
    
    # Apply Weight of Evidence encoding (converts all categorical features to numeric)
    ('WOEEncoder', WOEEncoder()),
]

# Build and fit the pipeline
pipe = Pipeline(steps=steps, verbose=True)
X_train_transformed = pipe.fit_transform(X_train, y_train)

print(f"\nOriginal features: {X_train.shape[1]}")
print(f"Engineered features: {X_train_transformed.shape[1]}")

[Pipeline] fit+transform   1/17 · IsNull  |  in: rows=891  cols=10  nulls=866  →  out: rows=891  cols=12  nulls=866  (0.001s)
[Pipeline] fit+transform   2/17 · Length  |  in: rows=891  cols=12  nulls=866  →  out: rows=891  cols=13  nulls=866  (0.001s)
[Pipeline] fit+transform   3/17 · SplitExtractName  |  in: rows=891  cols=13  nulls=866  →  out: rows=891  cols=13  nulls=866  (0.003s)
[Pipeline] fit+transform   4/17 · SplitExtractTitle  |  in: rows=891  cols=13  nulls=866  →  out: rows=891  cols=13  nulls=866  (0.001s)
[Pipeline] fit+transform   5/17 · MathFeatures  |  in: rows=891  cols=13  nulls=866  →  out: rows=891  cols=14  nulls=866  (0.000s)
[Pipeline] fit+transform   6/17 · ScalarMathFeatures  |  in: rows=891  cols=14  nulls=866  →  out: rows=891  cols=15  nulls=866  (0.000s)
[Pipeline] fit+transform   7/17 · ExtractSubstring  |  in: rows=891  cols=15  nulls=866  →  out: rows=891  cols=16  nulls=1553  (0.001s)
[Pipeline] fit+transform   8/17 · RenameColumns  |  in: rows=891  co

## 4. Generate Candidate Rules

Use XGBoost-based grid search to generate candidate rules from the engineered features. The `rule_grid_search_parallel_scales` function trains models with different `scale_pos_weight` values and extracts rules from the decision trees.

In [4]:
estimator = XGBClassifier(n_estimators=100, max_depth=4, eval_metric='logloss', random_state=0)
rules = rule_grid_search_parallel_scales(estimator, X_train_transformed, y_train, scale_pos_weight_vec=np.logspace(0, 3, 50))

In [5]:
print(f'Number of rules generated: {len(rules)}')

Number of rules generated: 1971


## 5. Select High-Quality Rules

Apply the generated rules to the training data, compute performance metrics, and filter based on:
- Minimum precision (> 0.15)
- Minimum recall (> 0.15)
- Maximum correlation between rules (< 0.8)

This ensures we keep only the most useful and diverse rules.

In [6]:
R = apply_rules(X_train_transformed, rules.select("rule").to_series().to_list())
M = compute_metrics(R, y_train)
M = M.filter((pl.col("precision") > 0.15) & (pl.col("recall") > 0.15)).sort("accuracy", descending=True)
importance = dict(zip(M["rule"], M["f0.5"]))
uncorrelated_rules = filter_correlated_rules(R[M['rule'].to_list()], importance=importance, max_corr=0.8)

In [7]:
num_rules = len(uncorrelated_rules)
print(f"Number of selected rules: {num_rules}")

Number of selected rules: 33


## 6. Combine Rules

Test different rule combination strategies to find the best performing ruleset.

### 6.1 Cumulative Combination

Combines rules cumulatively (rule1 OR rule2 OR ... OR ruleN):

In [8]:
R_combined = combine_rules_cumulative(R[uncorrelated_rules], output_names=[f"combined_rule_{i}" for i in range(1, num_rules+1)])
M_combined = compute_metrics(R_combined, y_train).sort("accuracy", descending=True)
M_combined.head(3)

rule,TP,FP,TN,FN,precision,recall,accuracy,flagged(%),good_flagged(%),f0.25,f0.5,f1,f1.5,f2,num_rules
str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""combined_rule_4""",252,57,492,90,0.815534,0.736842,0.835017,34.680135,10.382514,0.810443,0.798479,0.774194,0.759388,0.751342,1
"""combined_rule_5""",252,59,490,90,0.810289,0.736842,0.832772,34.904602,10.746812,0.805566,0.794451,0.771822,0.757982,0.750447,1
"""combined_rule_6""",252,59,490,90,0.810289,0.736842,0.832772,34.904602,10.746812,0.805566,0.794451,0.771822,0.757982,0.750447,1


### 6.2 Greedy Search

Uses a greedy algorithm to iteratively select the best rule combination:

In [ ]:
R_greedy = combine_rules_greedy(R[uncorrelated_rules], y_train, metric="accuracy")
M_greedy = compute_metrics(R_greedy, y_train)
M_greedy

rule,TP,FP,TN,FN,precision,recall,accuracy,flagged(%),good_flagged(%),f0.25,f0.5,f1,f1.5,f2,num_rules
str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""((X[""Title""] >= 0.25029) & (X[…",277,63,486,65,0.814706,0.809942,0.856341,38.159371,11.47541,0.814424,0.813749,0.812317,0.811402,0.81089,5


### 6.3 Beam Search

Uses beam search to explore rule combinations up to a maximum number of rules:

In [17]:
R_beam = combine_rules_beam_search(R[uncorrelated_rules], y_train, metric="accuracy", max_rules=10)
M_beam = compute_metrics(R_beam, y_train)
M_beam.head(3)

rule,TP,FP,TN,FN,precision,recall,accuracy,flagged(%),good_flagged(%),f0.25,f0.5,f1,f1.5,f2,num_rules
str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""((X[""Title""] >= 0.25029) & (X[…",255,58,491,87,0.814696,0.745614,0.837262,35.129068,10.564663,0.81028,0.799875,0.778626,0.765589,0.758477,4
"""((X[""Title""] >= 0.25029) & (X[…",255,58,491,87,0.814696,0.745614,0.837262,35.129068,10.564663,0.81028,0.799875,0.778626,0.765589,0.758477,5
"""((X[""Title""] >= 0.25029) & (X[…",255,58,491,87,0.814696,0.745614,0.837262,35.129068,10.564663,0.81028,0.799875,0.778626,0.765589,0.758477,5


### 6.3 A*

Use the A* algorithm to explore rule combinations up to a maximum number of rules.

In [ ]:
R_a_star = combine_rules_a_star(R[uncorrelated_rules], y_train, metric="accuracy", max_rules=10)
M_a_star = compute_metrics(R_a_star, y_train)
M_a_star.head(3)

## 7. Analyze the Best Ruleset

Generate a detailed report for the best performing ruleset from brute force combination:

In [24]:
ruleset = M_beam["rule"][0]
print(f"Selected ruleset: {ruleset}")
report = generate_rule_performance_report(ruleset, X_train_transformed, y_train)
report

Selected ruleset: ((X["Title"] >= 0.25029) & (X["FamilySize"] < 5.0)) | ((X["FarePerPerson_div"] >= 9.5) & (X["Sex"] >= 1.52977) & (X["Ticket__length"] >= 5.0) & (X["Fare"] < 151.55)) | ((X["Title"] >= 0.77539) & (X["FamilySize"] < 8.0) & (X["Pclass"] >= 0.36447)) | ((X["Title"] >= 0.77539) & (X["Fare"] >= 31.3875) & (X["Fare"] < 151.55) & (X["Ticket__length"] < 7.0))


rule_index,rule,TP,FP,TN,FN,precision,recall,accuracy,flagged(%),good_flagged(%),f0.25,f0.5,f1,f1.5,f2,num_rules
str,str,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32
"""0""","""((X[""Title""] >= 0.25029) & (X[…",255,58,491,87,0.814696,0.745614,0.837262,35.129068,10.564663,0.81028,0.799875,0.778626,0.765589,0.758477,4
"""0.0""","""(X['Title'] >= 0.25029) & (X['…",239,57,492,103,0.807432,0.69883,0.820426,33.2211,10.382514,0.800118,0.783093,0.749216,0.729,0.718149,1
"""0.1""","""(X['FarePerPerson_div'] >= 9.5…",127,6,543,215,0.954887,0.371345,0.751964,14.927048,1.092896,0.874089,0.726545,0.534737,0.457341,0.423051,1
"""0.2""","""(X['Title'] >= 0.77539) & (X['…",166,9,540,176,0.948571,0.48538,0.792368,19.640853,1.639344,0.898154,0.796545,0.642166,0.571202,0.537913,1
"""0.3""","""(X['Title'] >= 0.77539) & (X['…",59,1,548,283,0.983333,0.172515,0.681257,6.734007,0.182149,0.770353,0.506873,0.293532,0.231163,0.206583,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""0.2.2""","""(X['Pclass'] >= 0.36447)""",223,177,372,119,0.5575,0.652047,0.667789,44.893378,32.240437,0.562296,0.57415,0.601078,0.619709,0.630656,1
"""0.3.0""","""(X['Title'] >= 0.77539)""",249,98,451,93,0.717579,0.72807,0.785634,38.945006,17.850638,0.718188,0.719653,0.722787,0.72481,0.725948,1
"""0.3.1""","""(X['Fare'] >= 31.3875)""",129,86,463,213,0.6,0.377193,0.664422,24.130191,15.664845,0.579852,0.536606,0.463196,0.425851,0.407454,1


## 8. Generate Predictions on Test Data

Apply the same preprocessing pipeline to the test data, then use the best ruleset to generate predictions:

In [25]:
X_test = pl.read_csv('../../../kaggle/titanic/test.csv')
X_test_transformed = pipe.transform(X_test)
y_pred = eval(ruleset.replace("X", "X_test_transformed"))

[Pipeline] transform   1/17 · IsNull  |  in: rows=418  cols=11  nulls=414  →  out: rows=418  cols=13  nulls=414  (0.001s)
[Pipeline] transform   2/17 · Length  |  in: rows=418  cols=13  nulls=414  →  out: rows=418  cols=14  nulls=414  (0.000s)
[Pipeline] transform   3/17 · SplitExtractName  |  in: rows=418  cols=14  nulls=414  →  out: rows=418  cols=14  nulls=414  (0.001s)
[Pipeline] transform   4/17 · SplitExtractTitle  |  in: rows=418  cols=14  nulls=414  →  out: rows=418  cols=14  nulls=414  (0.001s)
[Pipeline] transform   5/17 · MathFeatures  |  in: rows=418  cols=14  nulls=414  →  out: rows=418  cols=15  nulls=414  (0.000s)
[Pipeline] transform   6/17 · ScalarMathFeatures  |  in: rows=418  cols=15  nulls=414  →  out: rows=418  cols=16  nulls=414  (0.000s)
[Pipeline] transform   7/17 · ExtractSubstring  |  in: rows=418  cols=16  nulls=414  →  out: rows=418  cols=17  nulls=741  (0.000s)
[Pipeline] transform   8/17 · RenameColumns  |  in: rows=418  cols=17  nulls=741  →  out: rows=41

In [26]:
# Create submission file (Kaggle leaderboard score: 0.76076)
# Note: Significantly better than without feature engineering (0.6411)
pl.DataFrame({"PassengerId": X_test["PassengerId"], "Survived": y_pred}).with_columns(pl.col("Survived").cast(pl.Int64)).write_csv("submission_titanic.csv")